# CLaP state-occurrence object study

Generated from `clap_researcher_input.ipynb`. CLaP remains the state detector.
This notebook preserves its output as observation-, event-, object-, and
relation-level tables without changing the inferred labels.


In [ ]:
from importlib.metadata import version

from claspy.data_loader import load_tssb_dataset
from claspy.state_detection import AgglomerativeCLaPDetection
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score

DATASET_NAME = "Crop"
EXPECTED_CLASPY_VERSION = "0.2.8"


def load_crop():
    row = load_tssb_dataset(names=(DATASET_NAME,)).iloc[0]
    dataset, window_size, true_cps, reference_states, signal = row
    return {
        "dataset": dataset,
        "window_size": int(window_size),
        "true_change_points": np.asarray(true_cps, dtype=int),
        "reference_states": np.asarray(reference_states),
        "signal": np.asarray(signal, dtype=float),
    }


def run_clap(signal):
    detector = AgglomerativeCLaPDetection()
    clap_states = np.asarray(detector.fit_predict(signal))
    sparse_states, sparse_transitions = detector.predict(sparse=True)
    return detector, clap_states, set(sparse_states), set(sparse_transitions)


def construct_observations(signal, reference_states, clap_states):
    if not (len(signal) == len(reference_states) == len(clap_states)):
        raise ValueError("Signal, reference states, and CLaP states must align.")
    observations = pd.DataFrame({
        "sample_index": np.arange(len(signal), dtype=int),
        "signal_raw": signal.copy(),
        "reference_state": reference_states.copy(),
        "clap_state": clap_states.copy(),
    })
    observations["enter_clap_state"] = observations["clap_state"].ne(
        observations["clap_state"].shift(1)
    )
    observations["exit_clap_state"] = observations["clap_state"].ne(
        observations["clap_state"].shift(-1)
    )
    observations["clap_change_point"] = (
        observations["enter_clap_state"] & observations["sample_index"].gt(0)
    )
    observations["occurrence_id"] = (
        observations["enter_clap_state"].cumsum().astype(int) - 1
    )
    return observations


def summarize_occurrences(observations):
    objects = (
        observations.groupby("occurrence_id", sort=True)
        .agg(
            clap_state=("clap_state", "first"),
            unique_state_count=("clap_state", "nunique"),
            start_index=("sample_index", "min"),
            end_index=("sample_index", "max"),
            duration_samples=("sample_index", "size"),
            signal_minimum=("signal_raw", "min"),
            signal_maximum=("signal_raw", "max"),
            signal_mean=("signal_raw", "mean"),
            signal_std=("signal_raw", lambda values: values.std(ddof=0)),
            enter_event_count=("enter_clap_state", "sum"),
            exit_event_count=("exit_clap_state", "sum"),
        )
        .reset_index()
    )
    objects["end_index_exclusive"] = objects["end_index"] + 1
    objects["previous_state"] = objects["clap_state"].shift(1)
    objects["next_state"] = objects["clap_state"].shift(-1)
    objects["boundary_fragment"] = False
    if not objects.empty:
        objects.loc[objects.index[[0, -1]], "boundary_fragment"] = True
    objects["is_complete"] = ~objects["boundary_fragment"]
    objects["object_label"] = objects["occurrence_id"].map(
        lambda value: f"CLAP-CROP-{int(value):03d}"
    )
    return objects


def construct_relations(objects):
    if len(objects) < 2:
        return pd.DataFrame(columns=[
            "relation", "source_occurrence_id", "target_occurrence_id",
            "source_state", "target_state", "boundary_index",
        ])
    relations = pd.DataFrame({
        "relation": "precedes",
        "source_occurrence_id": objects["occurrence_id"].iloc[:-1].to_numpy(),
        "target_occurrence_id": objects["occurrence_id"].iloc[1:].to_numpy(),
        "source_state": objects["clap_state"].iloc[:-1].to_numpy(),
        "target_state": objects["clap_state"].iloc[1:].to_numpy(),
        "boundary_index": objects["start_index"].iloc[1:].to_numpy(),
    })
    return relations


def compare_boundaries(predicted_change_points, true_change_points):
    if len(predicted_change_points) == len(true_change_points):
        comparison = pd.DataFrame({
            "boundary_number": np.arange(1, len(true_change_points) + 1),
            "reference_index": true_change_points,
            "clap_index": predicted_change_points,
        })
        comparison["signed_error_samples"] = (
            comparison["clap_index"] - comparison["reference_index"]
        )
        comparison["absolute_error_samples"] = comparison[
            "signed_error_samples"
        ].abs()
        comparison["matching_rule"] = "ordered_equal_count"
        return comparison
    raise ValueError(
        "Ordered boundary comparison requires equal counts; declare a matching rule first."
    )


def validate_study(source, observations, objects, relations, sparse_transitions):
    reconstructed = np.concatenate([
        np.repeat(row.clap_state, row.duration_samples)
        for row in objects.itertuples(index=False)
    ])
    predicted_cps = observations.loc[
        observations["clap_change_point"], "sample_index"
    ].to_numpy()
    relation_pairs = set(map(tuple, relations[["source_state", "target_state"]].to_numpy()))
    checks = {
        "raw_signal_preserved": np.array_equal(observations["signal_raw"].to_numpy(), source["signal"]),
        "one_label_per_observation": observations["clap_state"].notna().all(),
        "one_state_per_occurrence": objects["unique_state_count"].eq(1).all(),
        "occurrences_cover_source": int(objects["duration_samples"].sum()) == len(observations),
        "occurrence_ids_consecutive": objects["occurrence_id"].tolist() == list(range(len(objects))),
        "state_sequence_reconstructed": np.array_equal(reconstructed, observations["clap_state"].to_numpy()),
        "change_points_match_entries": np.array_equal(predicted_cps, objects["start_index"].iloc[1:].to_numpy()),
        "one_relation_per_adjacency": len(relations) == max(0, len(objects) - 1),
        "relations_match_sparse_clap_graph": relation_pairs == set(map(tuple, sparse_transitions)),
        "edge_fragments_retained": int(objects["boundary_fragment"].sum()) == 2,
    }
    report = pd.Series(checks, name="passed").rename_axis("check").to_frame()
    failed = report.index[~report["passed"]].tolist()
    if failed:
        raise AssertionError(f"CLaP object study validation failed: {failed}")
    return report


In [ ]:
source = load_crop()
detector, clap_states, sparse_states, sparse_transitions = run_clap(source["signal"])
observations = construct_observations(
    source["signal"], source["reference_states"], clap_states
)
state_occurrences = summarize_occurrences(observations)
occurrence_relations = construct_relations(state_occurrences)
predicted_change_points = observations.loc[
    observations["clap_change_point"], "sample_index"
].to_numpy()
boundary_comparison = compare_boundaries(
    predicted_change_points, source["true_change_points"]
)
validation_report = validate_study(
    source, observations, state_occurrences, occurrence_relations,
    sparse_transitions,
)

provenance = {
    "dataset": source["dataset"],
    "observations": len(observations),
    "window_size_from_benchmark": source["window_size"],
    "claspy_version": version("claspy"),
    "detector": type(detector).__name__,
    "detector_configuration": "defaults",
    "inferred_state_classes": sorted(map(int, sparse_states)),
    "sparse_transitions": sorted(map(lambda pair: tuple(map(int, pair)), sparse_transitions)),
}

summary = pd.Series({
    "observations": len(observations),
    "reference_change_points": len(source["true_change_points"]),
    "clap_change_points": len(predicted_change_points),
    "state_occurrences": len(state_occurrences),
    "complete_internal_occurrences": int(state_occurrences["is_complete"].sum()),
    "boundary_fragments": int(state_occurrences["boundary_fragment"].sum()),
    "adjacent_relations": len(occurrence_relations),
    "adjusted_rand_index": adjusted_rand_score(source["reference_states"], clap_states),
    "adjusted_mutual_information": adjusted_mutual_info_score(source["reference_states"], clap_states),
    "median_absolute_boundary_error_samples": boundary_comparison["absolute_error_samples"].median(),
    "maximum_absolute_boundary_error_samples": boundary_comparison["absolute_error_samples"].max(),
}, name="value").rename_axis("measure").to_frame()

display(pd.Series(provenance, name="value").rename_axis("field").to_frame())
display(validation_report)
display(summary)
display(boundary_comparison)
display(state_occurrences)
display(occurrence_relations)


## Interpretation limits

CLaP supplies the inferred state sequence. FeatureGraph does not alter or
improve those labels; it materializes each maximal run as an occurrence object
and preserves adjacency relations and provenance. State-class integers are
nominal identifiers rather than crop meanings. The first and final occurrences
are retained as series-boundary fragments because one outer transition is not
observed. Results from this one benchmark series do not establish general
interoperability.
